**TEST SECTION**

In [ ]:
model_path = 'models/rooftop_models_50epochs/best_model_50_epoch.h5'


In [ ]:
!pip install tensorflow opencv-python rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 83.4 MB/s eta 0:00:00


In [ ]:

from tensorflow.keras.models import load_model

# Load the saved model from Google Drive
model = load_model(model_path)

print(" Model loaded successfully!")


 Model loaded successfully!


In [ ]:
# Print model summary to confirm
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 512, 512, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 256, 256, 64)   │        32,832 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 256, 256, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 512, 512, 32)   │         8,224 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 512, 512, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 512, 512, 2)    │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,548 (705.27 KB)

 Trainable params: 180,546 (705.26 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
import os
BASE_DIR = '.'
IMAGE_DIR = f'{BASE_DIR}/image'
LABEL_DIR = f'{BASE_DIR}/labels'
TXT_PATH = f'{BASE_DIR}/train.txt'
OUTPUT_DIR = f'{BASE_DIR}/rooftop_predictions_50epochs'

print("Image Folder:", IMAGE_DIR)
print("Label Folder:", LABEL_DIR)
print("Text File:", TXT_PATH)
print("Output Folder", OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

Image Folder: /content/drive/MyDrive/rooftop_bigdata/image
Label Folder: /content/drive/MyDrive/rooftop_bigdata/labels
Text File: /content/drive/MyDrive/rooftop_bigdata/train.txt
Output Folder /content/drive/MyDrive/rooftop_bigdata/rooftop_predictions_50epochs


In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as pl
import os
import random
# Read image-label pairs
with open(TXT_PATH, "r") as f:
    lines = f.readlines()

# Shuffle and pick 60 random samples
random.shuffle(lines)
test_lines = lines[:60]

# Loop through 60 images
for idx, line in enumerate(test_lines):
    name = line.strip().split(';')
    image_path = os.path.join(IMAGE_DIR, name[0])
    label_path = os.path.join(LABEL_DIR, name[1])

    # Load and preprocess input image
    img_src = rasterio.open(image_path)
    img = np.moveaxis(img_src.read()[0:3], 0, 2) / 255.0
    img_input = np.expand_dims(img, axis=0)

    # Predict mask
    pred = model.predict(img_input)[0]
    pred_mask = np.argmax(pred, axis=-1).astype(np.uint8) * 255

    # Load ground truth mask
    label_src = rasterio.open(label_path)
    label = label_src.read(1).astype(np.uint8) * 255

    # Plot and save comparison
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(label, cmap='gray')
    ax[0].set_title("Ground Truth")
    ax[0].axis('off')

    ax[1].imshow(pred_mask, cmap='gray')
    ax[1].set_title("Predicted Mask")
    ax[1].axis('off')

    plt.tight_layout()
    comparison_path = os.path.join(OUTPUT_DIR, f'compare_{idx+1:03d}.png')
    plt.savefig(comparison_path, bbox_inches='tight')
    plt.close()

print("✅ Comparison images saved to:", OUTPUT_DIR)

/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 808ms/step


/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 596ms/step


/usr/local/lib/python3.11/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


✅ Comparison images saved to: /content/drive/MyDrive/rooftop_bigdata/rooftop_predictions_50epochs


## **comparison for 10 epoch and 50 epochs**

In [ ]:
import os

label_path = "data/labels" #add yours
print(sorted(os.listdir(label_path))[:10])  # show first 10 files


['output_1.png', 'output_10.png', 'output_100.png', 'output_1000.png', 'output_10000.png', 'output_10001.png', 'output_10002.png', 'output_10003.png', 'output_10004.png', 'output_10005.png']


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import load_model

# Paths
base_path = "."
img_path = os.path.join(base_path, "rooftop_bigdata/image")
label_path = os.path.join(base_path, "rooftop_bigdata/labels")
model_10_path = os.path.join(base_path, "rooftop_models_10epochs/final_model_10_epoch.h5")
model_50_path = os.path.join(base_path, "rooftop_models_50epochs/final_model_50_epoch.h5")
output_folder = os.path.join(base_path, "comparison_gt_epoch10_epoch50")

os.makedirs(output_folder, exist_ok=True)

# Load models
model10 = load_model(model_10_path)
model50 = load_model(model_50_path)

# Get sorted image list
image_files = sorted(os.listdir(img_path))[:60]

for i, filename in enumerate(image_files):
    # Load input image and preprocess
    img = Image.open(os.path.join(img_path, filename)).convert('RGB')
    img_resized = np.array(img.resize((512, 512))) / 255.0
    input_img = np.expand_dims(img_resized, axis=0)

    # Load and resize label
    label = Image.open(os.path.join(label_path, filename)).convert('L')
    label_arr = np.array(label.resize((256, 256)))

    # Predict
    pred10 = model10.predict(input_img)[0, :, :, 0]
    pred50 = model50.predict(input_img)[0, :, :, 0]

    # Threshold predictions and invert to make buildings=0, background=255
    pred10_mask = (pred10 <= 0.5).astype(np.uint8) * 255
    pred50_mask = (pred50 <= 0.5).astype(np.uint8) * 255

    # Label mask: no inversion — your GT is already black buildings
    label_mask = (label_arr > 127).astype(np.uint8) * 255  # black = 0, white = 255

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(label_mask, cmap='gray')
    axes[0].set_title("Ground Truth")

    axes[1].imshow(pred10_mask, cmap='gray')
    axes[1].set_title("Prediction - Epoch 10")

    axes[2].imshow(pred50_mask, cmap='gray')
    axes[2].set_title("Prediction - Epoch 50")

    for ax in axes:
        ax.axis('off')

    # Save result
    save_path = os.path.join(output_folder, f"comparison_{i+1:02d}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

print(f"\n✅ All {len(image_files)} comparison images saved to:\n{output_folder}")





1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 546ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 531ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 454ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 424ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 426ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 685ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 747ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 745ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 454ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 449ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 731ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# Set paths
base_path = '.'
label_path = os.path.join(base_path, 'labels')
pred10_path = os.path.join(base_path, 'rooftop_predictions_10epochs')
pred50_path = os.path.join(base_path, 'rooftop_predictions_50epochs')

# Folder to save comparison images
output_folder = os.path.join(base_path, 'comparison_gt_10_50')
os.makedirs(output_folder, exist_ok=True)

# Number of images to compare
num_images = 10

for i in range(num_images):
    filename = f"output_{i+1}.png"
    label_img = Image.open(os.path.join(label_path, filename))
    pred10_img = Image.open(os.path.join(pred10_path, filename))
    pred50_img = Image.open(os.path.join(pred50_path, filename))

    # Plot row with GT, Epoch 10, Epoch 50
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(label_img)
    axes[0].set_title("Ground Truth")
    axes[1].imshow(pred10_img)
    axes[1].set_title("Pred - Epoch 10")
    axes[2].imshow(pred50_img)
    axes[2].set_title("Pred - Epoch 50")

    for ax in axes:
        ax.axis('off')

    # Save the figure to Google Drive
    save_path = os.path.join(output_folder, f"comparison_{i+1:03d}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

print(f"Saved {num_images} comparison images to: {output_folder}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/rooftop_bigdata/rooftop_predictions_10epochs/output_1.png'

## **LIMITATION CHECK CODE**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import load_model

# Drive paths
base_path = "."
img_path = os.path.join(base_path, "rooftop_bigdata/image")
label_path = os.path.join(base_path, "rooftop_bigdata/labels")
model_10_path = os.path.join(base_path, "rooftop_models_10epochs/final_model_10_epoch.h5")
model_50_path = os.path.join(base_path, "rooftop_models_50epochs/final_model_50_epoch.h5")
output_folder = os.path.join(base_path, "model_limitation")

os.makedirs(output_folder, exist_ok=True)

# Load models
model10 = load_model(model_10_path)
model50 = load_model(model_50_path)

# Specific image numbers to analyze
target_ids = [36, 34, 31, 23, 71, 129, 208, 206]

# Loop over specific filenames
for id in target_ids:
    filename = f"output_{id}.png"

    # Full image and label paths
    img_file = os.path.join(img_path, filename)
    label_file = os.path.join(label_path, filename)

    # Check if image and label exist
    if not os.path.exists(img_file) or not os.path.exists(label_file):
        print(f"❌ Skipping {filename} (missing image or label)")
        continue

    # Load and preprocess image
    img = Image.open(img_file).convert('RGB')
    img_resized = np.array(img.resize((512, 512))) / 255.0
    input_img = np.expand_dims(img_resized, axis=0)

    # Load label and resize
    label = Image.open(label_file).convert('L')
    label_arr = np.array(label.resize((256, 256)))
    label_mask = (label_arr > 127).astype(np.uint8) * 255  # keep GT as building=black

    # Predict
    pred10 = model10.predict(input_img)[0, :, :, 0]
    pred50 = model50.predict(input_img)[0, :, :, 0]

    # Invert predictions to match GT format: buildings black (0), background white (255)
    pred10_mask = (pred10 <= 0.5).astype(np.uint8) * 255
    pred50_mask = (pred50 <= 0.5).astype(np.uint8) * 255

    # Plot 4-panel figure
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[1].imshow(label_mask, cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[2].imshow(pred10_mask, cmap='gray')
    axes[2].set_title("Prediction - Epoch 10")
    axes[3].imshow(pred50_mask, cmap='gray')
    axes[3].set_title("Prediction - Epoch 50")

    for ax in axes:
        ax.axis('off')

    # Save to model_limitation
    save_path = os.path.join(output_folder, f"limitation_{id}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

print(f"\n✅ Saved limitation images for {len(target_ids)} samples to: {output_folder}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 898ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 477ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 775ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 716ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 446ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step

✅ Saved limitation images for 8 samples to: /content/drive/MyDrive/model_limitation


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Directories
BASE_DIR = '.'
IMAGE_DIR = f'{BASE_DIR}/image'
LABEL_DIR = f'{BASE_DIR}/labels'
TXT_PATH = f'{BASE_DIR}/train.txt'

EPOCH10_DIR = f'{BASE_DIR}/rooftop_predictions_epoch10'
EPOCH50_DIR = f'{BASE_DIR}/rooftop_predictions_50epochs'
OUTPUT_DIR = f'{BASE_DIR}/comparison_epoch10_vs_50'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load file names
with open(TXT_PATH, "r") as f:
    lines = f.readlines()

# Select 5 images for visual comparison
selected_lines = lines[:5]

for idx, line in enumerate(selected_lines):
    name = line.strip().split(';')
    img_path = os.path.join(IMAGE_DIR, name[0])
    label_path = os.path.join(LABEL_DIR, name[1])

    # Load input image
    img_src = rasterio.open(img_path)
    img = np.moveaxis(img_src.read()[0:3], 0, 2) / 255.0

    # Load ground truth
    label_src = rasterio.open(label_path)
    label = label_src.read(1).astype(np.uint8) * 255

    # Load prediction masks
    pred10_path = os.path.join(EPOCH10_DIR, f'output_{idx}.png')  # Change if naming different
    pred50_path = os.path.join(EPOCH50_DIR, f'output_{idx}.png')

    pred10 = cv2.imread(pred10_path, cv2.IMREAD_GRAYSCALE)
    pred50 = cv2.imread(pred50_path, cv2.IMREAD_GRAYSCALE)

    # Plot all 4
    fig, ax = plt.subplots(1, 4, figsize=(18, 5))
    ax[0].imshow(img)
    ax[0].set_title("Original Image")
    ax[0].axis('off')

    ax[1].imshow(label, cmap='gray')
    ax[1].set_title("Ground Truth")
    ax[1].axis('off')

    ax[2].imshow(pred10, cmap='gray')
    ax[2].set_title("Prediction (Epoch 10)")
    ax[2].axis('off')

    ax[3].imshow(pred50, cmap='gray')
    ax[3].set_title("Prediction (Epoch 50)")
    ax[3].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'epoch_compare_{idx+1}.png'), bbox_inches='tight')
    plt.close()

print("✅ Comparison images saved to:", OUTPUT_DIR)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/rooftop_bigdata/train.txt'